# 09:45 SOXL/SOXX 双边首段来源门

仅使用原生 Qlib 15 分钟行情与冻结 Stage1 信号日。检查已完成 bar 与首次可用资格，不计算候选坐标或 SOXX 收益。

In [1]:
import hashlib, json, os, pathlib
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from qlib.workflow import R
root=pathlib.Path.cwd();card=root/'research/stage1_0945_soxl_soxx_cross_response_source_20261023'
plan=json.loads((card/'plan.json').read_text());src=plan['sources']
sha=lambda p:hashlib.sha256((root/p).read_bytes()).hexdigest()
assert sha(src['intraday_manifest'])==src['intraday_manifest_sha256']
assert sha(src['held_stage1_summary'])==src['held_stage1_summary_sha256']
assert sha(src['nas_authority_manifest'])==src['nas_authority_manifest_sha256']
manifest=json.loads((root/src['intraday_manifest']).read_text())
assert manifest['instruments']==['SOXL','SOXX'] and manifest['freq']=='15min'
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
qlib.init(provider_uri=str(root/src['intraday_provider']),region='us',kernels=1,expression_cache=None,dataset_cache=None)
calendar=pd.DatetimeIndex(D.calendar(start_time='2018-10-12',end_time='2026-10-07 16:00:00',freq='15min'))
cal_first=calendar[calendar.strftime('%H:%M')=='09:45']
fields=['$open','$close','$is_complete','$observed_1m_count','$expected_1m_count']
raw=D.features(['SOXL','SOXX'],fields,start_time='2018-10-12',end_time='2026-10-07 16:00:00',freq='15min')
frame=raw.reset_index();assert {'instrument','datetime',*fields}.issubset(frame.columns)
frame['datetime']=pd.to_datetime(frame['datetime']);first=frame.loc[frame.datetime.dt.strftime('%H:%M').eq('09:45')].copy()
first['signal_date']=first.datetime.dt.normalize()
assert not first.duplicated(['signal_date','instrument']).any()
first['valid']=np.isfinite(first['$open'])&np.isfinite(first['$close'])&(first['$open']>0)&(first['$close']>0)&first['$is_complete'].eq(1)&first['$observed_1m_count'].eq(15)&first['$expected_1m_count'].eq(15)
valid=first.pivot(index='signal_date',columns='instrument',values='valid').reindex(columns=['SOXL','SOXX']).fillna(False)
cal_dates=pd.DatetimeIndex(cal_first.normalize()).unique().sort_values();pair_dates=pd.DatetimeIndex(valid.index[valid.all(axis=1)]).sort_values()
assert set(pair_dates).issubset(set(cal_dates))
held_summary=json.loads((root/src['held_stage1_summary']).read_text())
with R.uri_context(held_summary['recorder_uri']):
 rec=R.get_recorder(recorder_id=held_summary['recorder_id'],experiment_id=held_summary['experiment_id'])
 assert rec.status=='FINISHED';daily=rec.load_object('analysis/daily')
held_dates=pd.DatetimeIndex(pd.to_datetime(daily.signal_date).dt.normalize());assert len(held_dates)==held_dates.nunique()==944
train_calendar=cal_dates[(cal_dates>=pd.Timestamp('2018-10-12'))&(cal_dates<=pd.Timestamp('2022-12-30'))]
train_pairs=pair_dates[(pair_dates>=pd.Timestamp('2018-10-12'))&(pair_dates<=pd.Timestamp('2022-12-30'))]
held_in_provider=held_dates.intersection(cal_dates);held_pairs=held_dates.intersection(pair_dates)
miss_held=held_dates.difference(held_pairs)
by_year={}
for year in sorted(set(held_dates.year)):
 dates=held_dates[held_dates.year==year];by_year[str(year)]={'stage1_signal_dates':len(dates),'paired_complete_first_bars':len(dates.intersection(pair_dates)),'missing_dates':[str(x.date()) for x in dates.difference(pair_dates)]}
result={'decision':'09_45_paired_complete_bar_source_coverage_only','plan_sha256':sha(card/'plan.json'),'provider_manifest_sha256':src['intraday_manifest_sha256'],'provider_calendar_firstbar_dates':len(cal_dates),'train_2018_2022_calendar_firstbar_dates':len(train_calendar),'train_2018_2022_complete_pair_dates':len(train_pairs),'train_2018_2022_missing_pair_dates':[str(x.date()) for x in train_calendar.difference(train_pairs)],'held_Stage1_signal_dates':len(held_dates),'held_dates_in_intraday_provider':len(held_in_provider),'held_complete_pair_dates':len(held_pairs),'held_missing_pair_dates':[str(x.date()) for x in miss_held],'held_by_year':by_year,'historical_first_receipt_qualified':False,'exact_NAS_snapshot_restore_verified_in_this_card':False,'cross_ETF_response_feature_qualified':False,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print(json.dumps({k:v for k,v in result.items() if k not in ('train_2018_2022_missing_pair_dates','held_missing_pair_dates','held_by_year')},ensure_ascii=False,indent=2))
print('held_missing_count',len(miss_held),'train_missing_count',len(train_calendar.difference(train_pairs)))


[59444:MainThread](2026-10-23 10:34:55,510) INFO - qlib.Initialization - [config.py:453] - default_conf: client.


[59444:MainThread](2026-10-23 10:34:55,512) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.


[59444:MainThread](2026-10-23 10:34:55,512) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl_intraday_15min')}


/Users/rziiiii/Library/Python/3.14/lib/python/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


{
  "decision": "09_45_paired_complete_bar_source_coverage_only",
  "plan_sha256": "bb8afbb17cdb9ac42bafb781300d49049e46f8baefdd4f45ecbe6667b9158057",
  "provider_manifest_sha256": "d373f95cb349532338e88160cbf9a1d56b75a68fdff3cea119ec96454bdc7c34",
  "provider_calendar_firstbar_dates": 2006,
  "train_2018_2022_calendar_firstbar_dates": 1062,
  "train_2018_2022_complete_pair_dates": 1061,
  "held_Stage1_signal_dates": 944,
  "held_dates_in_intraday_provider": 944,
  "held_complete_pair_dates": 944,
  "historical_first_receipt_qualified": false,
  "exact_NAS_snapshot_restore_verified_in_this_card": false,
  "cross_ETF_response_feature_qualified": false,
  "SOXX_outcome_reads": 0,
  "model_fits": 0,
  "native_accounts": 0
}
held_missing_count 0 train_missing_count 1


In [2]:
import hashlib,json,os,pathlib
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
card=pathlib.Path('research/stage1_0945_soxl_soxx_cross_response_source_20261023');plan=json.loads((card/'coordinate_plan.json').read_text())
base=json.loads((card/'result.json').read_text());assert base['train_2018_2022_complete_pair_dates']==1061 and base['held_complete_pair_dates']==944
root=pathlib.Path.cwd();os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/'data/qlib/soxl_intraday_15min'),region='us',kernels=1,expression_cache=None,dataset_cache=None)
cols=['$open','$close','$is_complete','$observed_1m_count','$expected_1m_count']
raw=D.features(['SOXL','SOXX'],cols,start_time='2018-10-12',end_time='2026-10-07 16:00:00',freq='15min').reset_index()
raw['datetime']=pd.to_datetime(raw.datetime)
first=raw.loc[raw.datetime.dt.strftime('%H:%M').eq('09:45')].copy();first['signal_date']=first.datetime.dt.normalize()
first['valid']=np.isfinite(first['$open'])&np.isfinite(first['$close'])&(first['$open']>0)&(first['$close']>0)&first['$is_complete'].eq(1)&first['$observed_1m_count'].eq(15)&first['$expected_1m_count'].eq(15)
first['response']=(first['$close']/first['$open']-1).where(first.valid)
mat=first.pivot(index='signal_date',columns='instrument',values='response').reindex(columns=['SOXL','SOXX'])
coordinate=mat.SOXL-3*mat.SOXX
assert coordinate.notna().sum()==len(first.loc[first.valid].groupby('signal_date').filter(lambda x:len(x)==2))/2
source=json.loads((root/'research/stage1_dual_task_39_information_20261023/plan.json').read_text())['source']
parent=root/source['parent_card']
with R.uri_context(str(parent/'qlib_recorder')):
 rec=R.get_recorder(experiment_id='905520356740140469',recorder_id=source['parent_recorder_id'])
 assert rec.status=='FINISHED';model=rec.load_object('models/base_outer_71');features=rec.load_object('inputs/features')
names=list(model.base_features);assert len(names)==39 and set(names).issubset(features.columns)
training=coordinate.loc[(coordinate.index>=pd.Timestamp('2018-10-12'))&(coordinate.index<=pd.Timestamp('2022-12-30'))].dropna()
common=features.loc[training.index,names]
corr=common.apply(lambda x:x.corr(training,method='spearman')).abs().sort_values(ascending=False)
assert len(training)==1061 and np.isfinite(training.to_numpy()).all() and len(corr)==39
held=coordinate.loc[(coordinate.index>=pd.Timestamp('2023-01-03'))&(coordinate.index<=pd.Timestamp('2026-10-07'))].dropna();assert len(held)==944
result={'plan_sha256':hashlib.sha256((card/'coordinate_plan.json').read_bytes()).hexdigest(),'formula':plan['formula'],'train_known_dates':len(training),'train_unknown_dates':base['train_2018_2022_calendar_firstbar_dates']-len(training),'held_known_dates':len(held),'train_quantiles':{str(q):float(training.quantile(q)) for q in (0,.01,.25,.5,.75,.99,1)},'max_abs_spearman_original39_train':float(corr.iloc[0]),'max_abs_spearman_original39_feature':str(corr.index[0]),'exact_duplicate_original_input':bool(corr.iloc[0]>=1-1e-12),'historical_first_receipt_qualified':False,'published_affine_price_basis_PIT_qualified':False,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'coordinate_result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print(json.dumps(result,ensure_ascii=False,indent=2))


{
  "plan_sha256": "d192071d57afb17f509b1a732aa976eab8c93c9831f95b9efb67803dc7493821",
  "formula": "(SOXL first completed 09:45 bar close/open - 1) - 3*(SOXX first completed 09:45 bar close/open - 1), from the same published-affine Qlib provider. It is a cross-ETF response, not ETF tracking error.",
  "train_known_dates": 1061,
  "train_unknown_dates": 1,
  "held_known_dates": 944,
  "train_quantiles": {
    "0": -0.04908931255340576,
    "0.01": -0.011667823791503907,
    "0.25": -0.002220332622528076,
    "0.5": -3.635883331298828e-05,
    "0.75": 0.0023293495178222656,
    "0.99": 0.00901559591293336,
    "1": 0.02712845802307129
  },
  "max_abs_spearman_original39_train": 0.15517445262625082,
  "max_abs_spearman_original39_feature": "range_pos_252",
  "exact_duplicate_original_input": false,
  "historical_first_receipt_qualified": false,
  "published_affine_price_basis_PIT_qualified": false,
  "SOXX_outcome_reads": 0,
  "model_fits": 0,
  "native_accounts": 0
}


## 中文结论

Qlib 09:45 双边完整 bar：2018–2022 为 1061/1062，2020-03-16 未知；冻结 Stage1 评价期 944/944。跨 ETF 首段响应与原39项最大训练期绝对 Spearman 0.1552，未见精确重复。历史首次接收、当前已发布复权价格的 PIT 以及精确 NAS 快照回读仍未合格。两次环境故障已保存收据。零预测标签、拟合或账户；只通过来源可构造门。
随后原始 NONE 分钟 NAS 快照远端哈希与清单定向回读通过；派生 Qlib bin 及逐日首次接收未证。来源回退筛选修复另见 `nas_source_readback.json`。